# 2. Baseline Evaluation: Zero-Shot vs Chain-of-Thought (CoT)
**HLT 2026 - Reasoning with Verifiable Rewards**

In questo notebook confrontiamo le prime due metodologie richieste dal progetto su **Qwen2.5-0.5B-Instruct** (o 1.5B):
1. **Metodo 1: Zero-Shot Diretto** (risposta secca senza passaggi).
2. **Metodo 2: Chain-of-Thought (CoT)** (richiesta esplicita di ragionamento passo-passo).

> ⚡ **Requisiti hardware**:
> * **Consigliato su Google Colab con GPU T4 gratuita** (il benchmark richiede circa 2-3 minuti).
> * Puo girare anche su CPU per 0.5B, ma e circa 5x piu lento.

## 1. Installazione librerie (se su Colab)

In [ ]:
!pip install -q transformers accelerate datasets pandas

## 2. Inizializzazione Ambiente & Verificatore AST

In [ ]:
import os
import ast
import re
import json
import torch
import pandas as pd
from typing import List, Dict, Tuple, Optional, Any
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilizzo dispositivo: {device}")
if device == "cuda":
    print("GPU rilevata:", torch.cuda.get_device_name(0))

In [ ]:
class CountdownVerifier:
    """
    Verificatore deterministico sicuro per il gioco Countdown basato su Python AST.
    """
    ALLOWED_OPERATORS = {
        ast.Add: lambda a, b: a + b,
        ast.Sub: lambda a, b: a - b,
        ast.Mult: lambda a, b: a * b,
        ast.Div: lambda a, b: a / b if b != 0 else None,
        ast.FloorDiv: lambda a, b: a // b if b != 0 else None,
        ast.USub: lambda a: -a,
        ast.UAdd: lambda a: +a
    }

    # Includiamo esplicitamente le classi operatore perche ast.walk le visita come nodi figli
    ALLOWED_NODE_TYPES = (
        ast.Expression,
        ast.BinOp,
        ast.UnaryOp,
        ast.Constant,
        ast.Add,
        ast.Sub,
        ast.Mult,
        ast.Div,
        ast.FloorDiv,
        ast.USub,
        ast.UAdd
    )

    @staticmethod
    def extract_tags(text: str) -> Tuple[Optional[str], Optional[str]]:
        """Estrae thought e answer dai tag XML-like <think> e <answer>."""
        think_match = re.search(r"<think>(.*?)</think>", text, re.DOTALL | re.IGNORECASE)
        thought = think_match.group(1).strip() if think_match else None

        answer_match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL | re.IGNORECASE)
        answer = answer_match.group(1).strip() if answer_match else None

        if not answer:
            answer_partial = re.search(r"<answer>(.*)", text, re.DOTALL | re.IGNORECASE)
            if answer_partial:
                answer = answer_partial.group(1).strip().split("\n")[0].strip()

        return thought, answer

    @classmethod
    def is_safe_ast(cls, tree: ast.AST) -> bool:
        """Verifica che l'albero contenga SOLO nodi e operatori aritmetici consentiti."""
        for node in ast.walk(tree):
            if not isinstance(node, cls.ALLOWED_NODE_TYPES):
                return False
            if isinstance(node, ast.BinOp) and type(node.op) not in cls.ALLOWED_OPERATORS:
                return False
            if isinstance(node, ast.UnaryOp) and type(node.op) not in cls.ALLOWED_OPERATORS:
                return False
            if isinstance(node, ast.Constant):
                # Esclude stringhe, None e booleani (in Python bool e sottoclasse di int)
                if isinstance(node.value, bool) or not isinstance(node.value, (int, float)):
                    return False
        return True

    @classmethod
    def evaluate_ast(cls, node: ast.AST) -> Optional[float]:
        """Valuta ricorsivamente un nodo AST consentendo solo operazioni matematiche di base."""
        if isinstance(node, ast.Expression):
            return cls.evaluate_ast(node.body)

        elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
            return float(node.value)

        elif isinstance(node, ast.UnaryOp):
            op_func = cls.ALLOWED_OPERATORS.get(type(node.op))
            if not op_func:
                return None
            val = cls.evaluate_ast(node.operand)
            return op_func(val) if val is not None else None

        elif isinstance(node, ast.BinOp):
            op_func = cls.ALLOWED_OPERATORS.get(type(node.op))
            if not op_func:
                return None
            left_val = cls.evaluate_ast(node.left)
            right_val = cls.evaluate_ast(node.right)
            if left_val is None or right_val is None:
                return None
            try:
                res = op_func(left_val, right_val)
                return res
            except (ZeroDivisionError, OverflowError):
                return None

        return None

    @classmethod
    def extract_numbers_from_ast(cls, node: ast.AST) -> List[int]:
        """Estrae tutti i numeri utilizzati nell'espressione AST."""
        numbers = []
        for n in ast.walk(node):
            if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
                if float(n.value).is_integer():
                    numbers.append(int(n.value))
                else:
                    numbers.append(n.value)
        return numbers

    @classmethod
    def verify(cls, available_numbers: List[int], target: int, completion: str) -> Dict[str, Any]:
        """
        Verifica completa del completamento generato dal modello.
        Restituisce un dizionario con lo stato di formattazione, validita e correttezza.
        """
        thought, answer_str = cls.extract_tags(completion)

        result = {
            "has_format": bool(thought is not None and answer_str is not None),
            "thought": thought,
            "answer_str": answer_str,
            "valid_syntax": False,
            "valid_numbers": False,
            "used_numbers": [],
            "computed_value": None,
            "is_correct": False,
            "error_msg": None
        }

        if not answer_str:
            result["error_msg"] = "Tag <answer> non trovato o vuoto"
            return result

        # 1. Parsing AST e Whitelist di sicurezza
        try:
            clean_expr = answer_str.split("=")[0].strip()
            tree = ast.parse(clean_expr, mode='eval')
            if not cls.is_safe_ast(tree):
                result["error_msg"] = "Espressione contiene elementi non consentiti (solo +, -, *, /, parentesi e numeri)"
                return result
            result["valid_syntax"] = True
        except Exception as e:
            result["error_msg"] = f"Errore sintassi AST: {e}"
            return result

        # 2. Controllo dei numeri utilizzati
        used_numbers = cls.extract_numbers_from_ast(tree)
        result["used_numbers"] = used_numbers

        available_copy = list(available_numbers)
        valid_nums = True
        for num in used_numbers:
            if num in available_copy:
                available_copy.remove(num)
            else:
                valid_nums = False
                break

        result["valid_numbers"] = valid_nums and (len(used_numbers) > 0)
        if not result["valid_numbers"]:
            result["error_msg"] = f"Numeri non validi o usati piu volte: {used_numbers} (Disponibili: {available_numbers})"
            return result

        # 3. Calcolo del valore
        computed = cls.evaluate_ast(tree)
        result["computed_value"] = computed

        if computed is None:
            result["error_msg"] = "Valutazione espressione fallita (es. divisione per zero)"
            return result

        # 4. Controllo del target
        if abs(computed - target) < 1e-6:
            result["is_correct"] = True
        else:
            result["error_msg"] = f"Risultato {computed} != target {target}"

        return result

## 3. Caricamento Modello Base (Qwen2.5-0.5B-Instruct)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

## 4. Caricamento Test Set

In [ ]:
test_path = "../data/test_countdown.jsonl"
if os.path.exists(test_path):
    with open(test_path, "r", encoding="utf-8") as f:
        test_samples = [json.loads(line) for line in f][:50]
else:
    print("Test set locale non trovato, utilizzo 3 campioni di esempio...")
    test_samples = [
        {"numbers": [84, 83, 34, 72], "target": 39},
        {"numbers": [25, 5, 2, 8], "target": 60},
        {"numbers": [12, 3, 4, 7], "target": 35}
    ]

print(f"Campioni selezionati per la valutazione: {len(test_samples)}")

## 5. Esecuzione Benchmark: Zero-Shot vs CoT

In [ ]:
def evaluate_prompt_mode(mode="zero_shot", samples=test_samples):
    correct_count = 0
    total_tokens = 0
    results = []

    for item in tqdm(samples, desc=f"Evaluating {mode}"):
        nums = item["numbers"]
        target = item["target"]

        if mode == "zero_shot":
            user_msg = f"Using numbers {nums}, create an equation that equals {target}. Do not explain. Output ONLY the equation in <answer>...</answer>."
        else: # Chain of Thought
            user_msg = f"Using numbers {nums}, create an equation that equals {target}. You can use (+, -, *, /) at most once per number. Think step by step inside <think>...</think> and output the final equation in <answer>...</answer>."

        messages = [
            {"role": "system", "content": "You are a helpful mathematical assistant."},
            {"role": "user", "content": user_msg}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=384 if mode == "cot" else 64,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        gen_tokens = output_ids[0][inputs.input_ids.shape[1]:]
        completion = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        total_tokens += len(gen_tokens)

        verif = CountdownVerifier.verify(nums, target, completion)
        if verif["is_correct"]:
            correct_count += 1

        results.append({
            "target": target,
            "numbers": nums,
            "completion": completion,
            "is_correct": verif["is_correct"],
            "tokens": len(gen_tokens)
        })

    acc = (correct_count / len(samples)) * 100
    avg_len = total_tokens / len(samples)
    print(f"\n--- Risultati {mode.upper()} ---")
    print(f"Accuratezza: {acc:.2f}% ({correct_count}/{len(samples)})")
    print(f"Lunghezza media completamento: {avg_len:.1f} token")
    return acc, avg_len, results

print("1. Valutazione Metodo 1: Zero-Shot Diretto...")
zs_acc, zs_len, zs_res = evaluate_prompt_mode("zero_shot")

print("\n2. Valutazione Metodo 2: Chain-of-Thought (CoT)...")
cot_acc, cot_len, cot_res = evaluate_prompt_mode("cot")

## 6. Risultati & Salvataggio Metriche

In [ ]:
df = pd.DataFrame([
    {"Metodologia": "1. Zero-Shot Direct", "Accuratezza (%)": f"{zs_acc:.2f}%", "Avg Tokens": f"{zs_len:.1f}"},
    {"Metodologia": "2. Chain-of-Thought (CoT)", "Accuratezza (%)": f"{cot_acc:.2f}%", "Avg Tokens": f"{cot_len:.1f}"}
])
print(df.to_string(index=False))

os.makedirs("../results", exist_ok=True)
with open("../results/baselines.json", "w") as f:
    json.dump({
        "zero_shot_accuracy": zs_acc,
        "zero_shot_tokens": zs_len,
        "cot_accuracy": cot_acc,
        "cot_tokens": cot_len
    }, f, indent=2)
print("Metriche salvate in results/baselines.json")